# GPU와 CPU 비교

출처: [CUDA 가이드](https://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html)

이 문서는 Triton 커널을 만들기 전에 알아야 할 GPU 프로그래밍의 기본 개념을 설명합니다. 대부분의 내용은 위에 링크한 CUDA 가이드에서 가져왔습니다.

CPU는 수십 개의 스레드를 병렬로 실행하며 주로 순차적인 연산을 수행합니다. GPU는 수천 개의 스레드를 병렬로 실행합니다. 덕분에 훨씬 빠르게 처리할 수 있지만, 프로그래밍 모델은 달라야 합니다.

![GPU 스레드](images/gpu_threads.png)

GPU 프로그래밍의 핵심은 데이터를 디바이스 메모리로 옮겨 병렬 처리한 다음 결과를 다시 가져오는 것입니다. GPU와 데이터를 주고받는 작업은 느립니다. FlashAttention과 같은 알고리즘은 데이터 복사를 최소화하고 연산 자원 활용률을 높입니다.

> 애플리케이션은 호스트와 디바이스 사이의 데이터 전송을 최소화해야 합니다. 이를 위한 한 가지 방법은 디바이스에서 충분한 병렬성을 발휘하지 못하는 커널이라도 호스트에서 디바이스로 더 많은 코드를 옮겨 실행하는 것입니다. 중간 데이터 구조는 디바이스 메모리에서 생성하고, 처리하고, 삭제할 수 있으며 이 과정에서 호스트로 매핑하거나 복사할 필요가 없습니다. 또한 각 전송에는 오버헤드가 있으므로, 작은 전송 여러 번보다 큰 전송 한 번으로 묶는 편이 항상 성능이 좋습니다.

# GPU 아키텍처

GPU는 스트리밍 멀티프로세서(Streaming Multiprocessor, SM) 배열을 중심으로 구성됩니다. 멀티스레드 프로그램은 서로 독립적으로 실행되는 스레드 블록으로 나뉩니다. 따라서 멀티프로세서가 더 많은 GPU는 멀티프로세서가 더 적은 GPU보다 프로그램을 더 짧은 시간에 실행할 수 있습니다.

NVIDIA GPU 아키텍처는 확장 가능한 멀티스레드 스트리밍 멀티프로세서 배열로 구성됩니다. 호스트 CPU에서 CUDA 프로그램이 커널 그리드를 호출하면, 그리드의 블록이 나열되어 실행 여유가 있는 멀티프로세서에 분배됩니다. 한 스레드 블록의 스레드는 하나의 멀티프로세서에서 동시에 실행되며, 하나의 멀티프로세서에서 여러 스레드 블록이 동시에 실행될 수도 있습니다. 스레드 블록이 종료되면 비어 있는 멀티프로세서에서 새 블록이 실행됩니다.

멀티프로세서는 32개의 병렬 스레드로 구성된 그룹인 워프(warp) 단위로 스레드를 생성하고 관리하며 스케줄링하고 실행합니다. 워프를 구성하는 각 스레드는 같은 프로그램 주소에서 시작하지만, 각자 명령 주소 카운터와 레지스터 상태를 가지므로 서로 독립적으로 분기하고 실행할 수 있습니다. 워프라는 이름은 병렬 스레드 기술의 초기 형태인 직조기에서 유래했습니다. 하프 워프는 워프의 앞쪽 절반 또는 뒤쪽 절반을 뜻하고, 쿼터 워프는 앞쪽부터 네 구간 중 하나를 뜻합니다.

멀티프로세서에 하나 이상의 스레드 블록이 할당되면 블록은 워프로 나뉘고, 각 워프는 워프 스케줄러가 실행하도록 예약됩니다. 블록을 워프로 나누는 방식은 항상 동일합니다. 각 워프에는 연속적으로 증가하는 스레드 ID를 가진 스레드가 들어가며, 첫 번째 워프에는 스레드 0이 포함됩니다.

# CUDA

CUDA를 사용하면 작업을 GPU에서 효율적으로 실행할 수 있는 작은 구성 요소로 나눌 수 있습니다. 수많은 스레드를 고려해야 하므로 코드를 구조화하는 방식도 달라집니다.

## 커널 그리드와 스레드 블록

CUDA C++는 프로그래머가 커널이라고 부르는 C++ 함수를 정의할 수 있도록 확장되었습니다. 커널을 호출하면 일반 C++ 함수처럼 한 번만 실행되는 것이 아니라, 서로 다른 CUDA 스레드 N개에서 병렬로 N번 실행됩니다.

커널은 여러 스레드 블록으로 구성할 수 있습니다. 다음은 1차원 스레드 블록의 예입니다.

```
// 커널 정의
__global__ void VecAdd(float* A, float* B, float* C)
{
    int i = threadIdx.x;
    C[i] = A[i] + B[i];
}

int main()
{
    ...
    // N개 스레드로 커널 호출
    VecAdd<<<1, N>>>(A, B, C);
    ...
}
```

편의를 위해 `threadIdx`는 3개 요소로 이루어진 벡터입니다. 따라서 스레드는 1차원, 2차원 또는 3차원 인덱스로 식별할 수 있으며, 이에 따라 1차원, 2차원 또는 3차원 블록을 구성할 수 있습니다. 이는 벡터, 행렬 또는 부피와 같은 데이터 영역의 각 요소에 대해 연산을 실행하는 자연스러운 방법입니다.

`threadIdx`를 입력 데이터의 한 개 이상의 차원에 대한 인덱스로 생각할 수 있습니다. 데이터를 여러 조각으로 나눈 뒤 각 조각을 개별적으로 처리하는 것입니다. 이렇게 하면 데이터 복사를 최소화하고 병렬성을 활용할 수 있습니다.

다음은 2차원 스레드 블록의 예입니다.

```
// 커널 정의
__global__ void MatAdd(float A[N][N], float B[N][N],
                       float C[N][N])
{
    int i = threadIdx.x;
    int j = threadIdx.y;
    C[i][j] = A[i][j] + B[i][j];
}

int main()
{
    ...
    // N * N * 1개 스레드로 블록 하나를 호출
    int numBlocks = 1;
    dim3 threadsPerBlock(N, N);
    MatAdd<<<numBlocks, threadsPerBlock>>>(A, B, C);
    ...
}
```

스레드 ID는 블록 안에서 유일합니다. 스레드의 인덱스와 ID 사이의 관계는 간단합니다. 1차원 블록에서는 두 값이 같습니다. 크기가 `(Dx, Dy)`인 2차원 블록에서 인덱스가 `(x, y)`인 스레드의 ID는 `(x + y Dx)`입니다. 크기가 `(Dx, Dy, Dz)`인 3차원 블록에서는 `(x + y Dx + z Dx Dy)`입니다.

블록의 모든 스레드는 같은 스트리밍 멀티프로세서 코어에 상주하고 제한된 코어 메모리 자원을 공유해야 하므로 블록당 스레드 수에는 한계가 있습니다. 현재 GPU에서는 스레드 블록 하나에 최대 1024개의 스레드를 포함할 수 있습니다.

하지만 커널은 모양이 같은 여러 스레드 블록으로 실행할 수 있습니다. 따라서 전체 스레드 수는 블록당 스레드 수에 블록 수를 곱한 값이 됩니다.

블록은 1차원, 2차원 또는 3차원 그리드로 구성합니다. 이를 보통 실행 그리드 또는 커널 그리드라고 합니다. 그리드의 블록 수는 대개 처리할 데이터 크기에 따라 정하며, 일반적으로 시스템의 프로세서 수보다 데이터가 많습니다.

![스레드 그리드](images/grid-of-thread-blocks.png)

그리드의 각 블록은 1차원, 2차원 또는 3차원의 고유 인덱스로 식별할 수 있으며, 커널 안에서 내장 변수 `blockIdx`를 통해 접근할 수 있습니다. 스레드 블록의 차원은 `blockDim` 변수를 통해 확인할 수 있습니다.

다음은 여러 스레드 블록으로 구성된 실행 그리드를 사용하는 행렬 덧셈 예제입니다.

```
// 커널 정의
__global__ void MatAdd(float A[N][N], float B[N][N],
float C[N][N])
{
    int i = blockIdx.x * blockDim.x + threadIdx.x; // 현재 블록 인덱스 * 블록 크기 + 블록 내 현재 스레드 인덱스
    int j = blockIdx.y * blockDim.y + threadIdx.y; // 현재 블록 인덱스 * 블록 크기 + 블록 내 현재 스레드 인덱스
    if (i < N && j < N)
        C[i][j] = A[i][j] + B[i][j]; // 스레드 블록과 블록 내부 인덱스를 이용해 요소에 접근
}

int main()
{
    ...
    // 커널 호출
    dim3 threadsPerBlock(16, 16);
    dim3 numBlocks(N / threadsPerBlock.x, N / threadsPerBlock.y);
    MatAdd<<<numBlocks, threadsPerBlock>>>(A, B, C);
    ...
}
```

입력 행렬을 블록 단위로 나눈 다음 각 블록을 개별적으로 처리하고, 블록 안에서는 요소별로 연산합니다.

이 예제에서는 스레드 블록 크기 16x16(256개 스레드)을 사용했지만, 이 값은 임의로 정한 것입니다. 각 행렬 요소마다 스레드 하나를 할당하려면 앞서 설명한 것처럼 그리드에 충분한 블록을 만들어야 합니다. 단순화를 위해 예제에서는 각 차원의 스레드 수가 해당 차원의 블록 크기로 나누어떨어진다고 가정합니다. 하지만 반드시 나누어떨어질 필요는 없습니다.

스레드 블록은 서로 독립적으로 실행되어야 합니다. 즉, 어떤 순서로든 병렬 또는 순차적으로 실행할 수 있어야 합니다. 이 독립성 덕분에 GPU 코어 수에 상관없이 프로그래머가 블록의 실행 순서를 지정하지 않아도 되며, 코어 수가 달라져도 확장 가능한 프로그램을 작성할 수 있습니다.

블록 안의 스레드는 일부 데이터를 공유 메모리에 저장하고 실행을 동기화하여 메모리 접근을 조율할 수 있습니다. 더 정확히는 `__syncthreads()` 내장 함수를 호출해 동기화 지점을 지정할 수 있습니다. `__syncthreads()`는 블록의 모든 스레드가 도달할 때까지 대기하는 장벽으로, 모든 스레드가 도착하기 전에는 어느 스레드도 다음 단계로 진행할 수 없습니다. 공유 메모리 예제에서는 이를 사용합니다. `__syncthreads()` 외에도 Cooperative Groups API가 다양한 스레드 동기화 기능을 제공합니다.

효율적인 협력을 위해 공유 메모리는 각 프로세서 코어 가까이에 있는 저지연 메모리(L1 캐시와 유사)로 설계되며, `__syncthreads()`도 가벼운 동기화 연산으로 동작합니다.

## 스레드 블록 클러스터

NVIDIA Compute Capability 9.0부터 CUDA 프로그래밍 모델에는 스레드 블록으로 구성된 선택적 계층인 스레드 블록 클러스터가 추가되었습니다. 하나의 스레드 블록에 속한 스레드가 같은 스트리밍 멀티프로세서에서 함께 스케줄링되는 것처럼, 한 클러스터의 스레드 블록도 GPU의 GPU Processing Cluster(GPC)에서 함께 스케줄링되도록 보장됩니다.

스레드 블록과 마찬가지로 클러스터도 1차원, 2차원 또는 3차원으로 구성할 수 있습니다(그림 5 참고). 클러스터에 포함되는 스레드 블록 수는 사용자가 지정할 수 있으며, CUDA에서 이식 가능한 클러스터 크기의 최댓값은 8개입니다. 다만 8개의 멀티프로세서를 지원하기에 너무 작은 GPU 하드웨어 또는 MIG 구성에서는 최댓값이 줄어듭니다. 클러스터 크기 8을 초과해 지원하는 더 큰 구성의 여부와 이런 작은 구성의 여부는 아키텍처별로 다르며 `cudaOccupancyMaxPotentialClusterSize` API로 조회할 수 있습니다.

![클러스터 그리드](images/grid-of-clusters.png)

# 메모리 접근

CUDA 프로그램을 최적화할 때 메모리 접근 최적화가 가장 중요합니다.

CUDA 스레드는 실행 중 여러 메모리 공간의 데이터에 접근할 수 있습니다. 각 스레드는 전용 로컬 메모리를 가집니다. 각 스레드 블록은 블록의 모든 스레드가 볼 수 있고 블록이 살아 있는 동안 유지되는 공유 메모리를 가집니다. 스레드 블록 클러스터 안의 블록들은 서로의 공유 메모리에 읽기, 쓰기 및 원자적 연산을 수행할 수 있습니다. 모든 스레드는 같은 전역 메모리에 접근할 수 있습니다.

![메모리 계층](images/memory-hierarchy.png)

모든 스레드가 접근할 수 있는 읽기 전용 메모리 공간으로 상수 메모리와 텍스처 메모리도 있습니다. 전역, 상수, 텍스처 메모리는 서로 다른 메모리 사용 방식에 맞게 최적화되어 있습니다(디바이스 메모리 접근 참고). 텍스처 메모리는 일부 특정 데이터 형식에 대해 별도의 주소 지정 모드와 데이터 필터링 기능도 제공합니다(텍스처 및 서피스 메모리 참고).

전역, 상수 및 텍스처 메모리 공간은 같은 애플리케이션에서 실행하는 커널 호출 사이에도 유지됩니다.

성능 최적화는 다음 네 가지 기본 전략을 중심으로 이루어집니다.
- 최대한 병렬로 실행해 GPU 활용률을 높입니다.
- 메모리 사용을 최적화해 메모리 처리량을 높입니다.
- 명령어 사용을 최적화해 명령어 처리량을 높입니다.
- 메모리 스래싱을 최소화합니다.

각 멀티프로세서에서 실행되는 워프의 프로그램 카운터, 레지스터 등의 실행 컨텍스트는 워프가 살아 있는 동안 온칩에 유지됩니다. 따라서 한 실행 컨텍스트에서 다른 컨텍스트로 전환하는 데 비용이 들지 않습니다. 각 명령어 발행 시점에 워프 스케줄러는 다음 명령어를 실행할 준비가 된 스레드가 있는 워프(활성 스레드가 있는 워프)를 선택해 해당 스레드에 명령어를 발행합니다.

각 멀티프로세서에는 워프에 분배되는 32비트 레지스터 집합과 스레드 블록에 분배되는 병렬 데이터 캐시 또는 공유 메모리가 있습니다.